# Direct Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Direct baseline for English-to-Chinese cross-lingual dialogue summarization.

The Direct pipeline uses a single local small language model agent. The agent reads the original English dialogue and directly generates a concise Chinese summary without using an intermediate English summary, full-dialogue translation, semantic representation, or revision step.

```text
English Dialogue
→ Direct Summarization Agent
→ Final Chinese Summary
```

The pipeline consists of one agent:

```text
Direct Summarization Agent
Input: original English dialogue
Output: concise Chinese summary
```
This setup is used as the simplest cross-lingual summarization baseline. Unlike Translate-then-Summarize, Summarize-then-Translate, or the Semantic-agent pipeline, the Direct pipeline performs the task in a single model call.

The local small language model is served through Ollama. The notebook controls the prompt design, input/output processing, direct summary generation, output inspection, and result saving.

## 1. Model Setup

This notebook is configured for qwen 3.5 (9.65B).

Ollama model page: https://ollama.com/library/qwen3.5:9b

Pull the model before running inference:

```bash
ollama pull qwen3.5:9b
```

The config cell below sets:

```python
DIRECT_MODEL = "qwen3.5:9b"
```


In [ ]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm


In [ ]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Direct model
DIRECT_MODEL = "qwen3.5:9b"  # Change this if your local Ollama model name is different

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192
EXPECTED_SAMPLE_COUNT = 100


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root from a notebook or project working directory."""
    for path in [start, *start.parents]:
        if (path / "data" / "splits" / "test_100_seed42.json").exists():
            return path
    raise FileNotFoundError("Could not find data/splits/test_100_seed42.json")


PROJECT_ROOT = find_project_root()

# Fixed 100-sample test set, shared with the mBART baseline
DATASET_PATH = PROJECT_ROOT / "data" / "splits" / "test_100_seed42.json"

# Output directory
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "direct" / "qwen3.5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint and final output files for Direct baseline
CHECKPOINT_JSONL_PATH = OUTPUT_DIR / "direct_qwen9b_100samples_seed42_checkpoint.jsonl"
FINAL_JSON_PATH = OUTPUT_DIR / "direct_qwen9b_100samples_seed42_results.json"
FINAL_CSV_PATH = OUTPUT_DIR / "direct_qwen9b_100samples_seed42_results.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "direct_qwen9b_100samples_seed42_errors.jsonl"

print("Dataset path:", DATASET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Checkpoint JSONL path:", CHECKPOINT_JSONL_PATH)
print("Final JSON output path:", FINAL_JSON_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)


In [ ]:
print(DATASET_PATH.exists())


In [ ]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

In [ ]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [ ]:
# Cell 5: Direct prompt template
Please summarize the following text in Chinese:

{dialogue}

In [ ]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [ ]:
# Cell 7: Direct agent function

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def direct_agent(dialogue: str) -> str:
    """Direct baseline: English dialogue -> Chinese summary."""
    prompt = fill_prompt(
        DIRECT_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=DIRECT_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [ ]:
# Cell 8: Direct pipeline

def run_direct_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the Direct pipeline: English dialogue -> Chinese summary."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    final_chinese_summary = direct_agent(dialogue)

    return {
        "id": sample_id,
        "sample_index": example.get("sample_index", ""),
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "final_summary": final_chinese_summary,
        "pipeline": "direct",
        "model": DIRECT_MODEL,
        "num_model_calls": 1,
    }

## 3. Test with Examples

Load the fixed 100-sample test set before running the full direct baseline.


In [ ]:
# Cell 9: Load fixed 100-sample test set

def load_examples_from_sample_set(path: Path) -> List[Dict[str, Any]]:
    """Load the fixed 100-example sample shared across baseline runs."""
    if not path.exists():
        raise FileNotFoundError(f"Sample set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    if not isinstance(raw_data, list):
        raise TypeError(f"Expected a top-level JSON array in {path}")

    examples = []

    for i, item in enumerate(raw_data):
        sample_index = item.get("sample_index", i)
        examples.append({
            "id": item.get("id", f"test100_seed42_{sample_index:05d}"),
            "sample_index": sample_index,
            "test_index": item.get("test_index", sample_index),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_sample_set(DATASET_PATH)

if len(test_data) != EXPECTED_SAMPLE_COUNT:
    raise ValueError(f"Expected {EXPECTED_SAMPLE_COUNT} examples, got {len(test_data)}")

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])


In [ ]:
print(DIRECT_PROMPT)

In [ ]:
# Cell 10: Run the Direct pipeline for the first example

result = run_direct_pipeline(test_data[4])
result

In [ ]:
# Cell 11: Print Direct pipeline result clearly

def print_direct_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Direct Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Model:", result["model"])
    print("Model calls:", result["num_model_calls"])


print_direct_result(result)

## 4. Save Results

This saves all intermediate outputs and the final output.


In [ ]:
# Cell 12: Reset previous outputs before batch inference

CHECKPOINT_JSONL_PATH.unlink(missing_ok=True)
FINAL_JSON_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("Checkpoint JSONL path:", CHECKPOINT_JSONL_PATH)
print("Final JSON output path:", FINAL_JSON_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)


## 5. Batch Inference with Checkpointing

This cell processes the fixed 100-sample dataset one example at a time and appends each completed result to the configured JSONL file.

If the notebook stops, already processed examples remain saved.


In [ ]:
# Cell 13: Batch inference with Direct pipeline

MAX_EXAMPLES = len(test_data)
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(CHECKPOINT_JSONL_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running Direct pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_direct_pipeline(ex)
        append_jsonl(record, CHECKPOINT_JSONL_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "sample_index": ex.get("sample_index", ""),
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Checkpoint outputs saved to: {CHECKPOINT_JSONL_PATH}")


## 6. Export Final Results to JSON and CSV

The final JSON and CSV outputs use the same schema as the mBART baseline: `sample_index`, `model_name`, `generated_summary_zh`, `reference_summary_zh`, `reference_summary_en`, and `dialogue`.


In [ ]:
# Cell 14: Export Direct summaries to JSON and CSV

records = load_jsonl(CHECKPOINT_JSONL_PATH)

OUTPUT_FIELDNAMES = [
    "sample_index",
    "model_name",
    "generated_summary_zh",
    "reference_summary_zh",
    "reference_summary_en",
    "dialogue",
]

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "sample_index": record.get("sample_index", ""),
        "model_name": record.get("model", DIRECT_MODEL),
        "generated_summary_zh": record.get("final_summary", ""),
        "reference_summary_zh": record.get("reference_chinese_summary", ""),
        "reference_summary_en": record.get("reference_english_summary", ""),
        "dialogue": record.get("dialogue", ""),
    })


def sample_sort_key(row: Dict[str, Any]) -> int:
    try:
        return int(row.get("sample_index", 10**9))
    except (TypeError, ValueError):
        return 10**9


df = pd.DataFrame(rows, columns=OUTPUT_FIELDNAMES)

if not df.empty:
    df = df.drop_duplicates(subset=["sample_index"], keep="last")
    df = df.sort_values(by="sample_index", key=lambda s: s.map(lambda value: sample_sort_key({"sample_index": value})))

json_records = df.to_dict(orient="records")

FINAL_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with FINAL_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(json_records, f, ensure_ascii=False, indent=2)
    f.write("\n")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8")

print(f"Saved final JSON results to: {FINAL_JSON_PATH}")
print(f"Saved final CSV results to: {FINAL_CSV_PATH}")
df


In [ ]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "sample_index",
    "generated_summary_zh",
    "reference_summary_zh",
]

comparison_df = df[comparison_columns].copy()

comparison_df


In [ ]:
# Cell 16: Inspect Direct outputs

if not df.empty:
    inspection_columns = [
        "sample_index",
        "model_name",
        "dialogue",
        "generated_summary_zh",
        "reference_summary_zh",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")
